# Music Annotation Project V8.1
## Full-Pixel, Whole-Page, Source-Group Out-of-Fold Evaluation

**Complete build with audited page-level fold-map handoff.**

This notebook:
- consumes a hash-verified PASS output from `V8_1_Fold_Map_Resolution_Audit.ipynb`;
- validates its page → source-group → held-out-fold map against the development manifest;
- searches the V8.0b/c/d research roots for frozen artefacts;
- reproduces frozen V7.8 correction arrays;
- enumerates every eligible pixel;
- performs batched source-group OOF inference;
- applies the registered PRINT / ANNOTATION / UNCERTAIN rule;
- assembles whole corrected pages;
- locks predictions before clean-truth scoring;
- calculates full-pixel metrics and creates a visual-audit queue.

**Fail closed:** no fold, schema, calibrator, checkpoint, OOD or ensemble object is refitted or invented.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib scipy scikit-learn joblib torch torchvision psutil


## 1. Configuration


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, os, sys, platform, time, re, warnings, inspect, shutil
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib, psutil
import torch
from torch import nn

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V75_ROOT=ROOT/'reconstruction_geometry_benchmark_v7_5'
V8_SEARCH_ROOTS=[ROOT/'observed_print_classifier_v8_0b',ROOT/'observed_print_classifier_v8_0c',ROOT/'observed_print_classifier_v8_0d']
OUTPUT_ROOT=ROOT/'observed_print_classifier_v8_1'
FOLD_AUDIT_ROOT=ROOT/'v8_1_full_pixel_oof'/'fold_map_audit'

V75_RUN_OVERRIDE=None
V8_RUN_OVERRIDE=None
FOLD_AUDIT_RUN_OVERRIDE=None
DEVELOPMENT_MANIFEST_OVERRIDE=None
SCHEMA_OVERRIDE=None
POLICY_OVERRIDE=None
REPLAY_OVERRIDE=None
CHECKPOINT_BY_FOLD_OVERRIDE={}
CALIBRATOR_BY_FOLD_OVERRIDE={}

MASK_DILATION=2; CROP_MARGIN=48; GEOMETRY_THRESHOLD=80; BAND_WIDTH=4
PATCH_SIZE=33; PATCH_RADIUS=16; OBSERVED_DARK_THRESHOLD=200
PRINT_THRESHOLD=0.995; ANNOTATION_THRESHOLD=0.995; PRINT_ANNOTATION_MARGIN=0.990; MIN_PRINT_RUN=2
BOOTSTRAP_SAMPLES=10000; BOOTSTRAP_SEED=8101
BATCH_SIZE=4096; DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}

RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_v8_1_full_pixel_oof'
OUT=OUTPUT_ROOT/RUN
DIRS={x:OUT/x for x in ['01_input_audit','02_v78_equivalence','03_eligibility','04_oof_predictions','05_prediction_lock','06_truth_scoring','07_whole_pages','08_visual_audit','09_replay','failures']}
OUT.mkdir(parents=True,exist_ok=False)
for p in DIRS.values(): p.mkdir(parents=True,exist_ok=False)
print('DEVICE:',DEVICE); print('OUTPUT:',OUT)


## 2. Integrity helpers


In [ ]:
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()
def object_hash(v): return hashlib.sha256(json.dumps(v,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()
def atomic_json(path,v):
    path=Path(path); tmp=Path(str(path)+'.tmp'); tmp.write_text(json.dumps(v,indent=2,sort_keys=True,default=str)); tmp.replace(path)
def fail(stage,message,details=None):
    rec={'created_utc':datetime.now(timezone.utc).isoformat(),'stage':stage,'message':message,'details':details}; atomic_json(DIRS['failures']/f'{stage}_failure.json',rec); raise RuntimeError(f'{stage}: {message}')
def dedupe(xs): return sorted(set(Path(x) for x in xs),key=lambda p:str(p))
def image_map(folder):
    folder=Path(folder); return {p.stem:p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if folder.exists() else {}
def latest_complete(root,required):
    hits=[]; root=Path(root)
    if root.exists():
        for p in root.rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None


## 3. Project-wide V8 artefact census


In [ ]:
V75=Path(V75_RUN_OVERRIDE) if V75_RUN_OVERRIDE else latest_complete(V75_ROOT,['tables/cache_status.csv'])
if V75 is None: fail('input_discovery','No complete V7.5 run found')
if V8_RUN_OVERRIDE is not None: V8_SEARCH_ROOTS.insert(0,Path(V8_RUN_OVERRIDE))
V8_SEARCH_ROOTS=[p for p in V8_SEARCH_ROOTS if p.exists()]
if not V8_SEARCH_ROOTS: fail('input_discovery','No V8.0b/c/d roots exist')
patterns={
'inventory':['artifact_inventory.json','artefact_inventory.json','*artifact*inventory*.json','*artefact*inventory*.json'],
'development_manifest':['development_score_manifest.csv','development_score_manifest.json','paired_sample_manifest.csv','paired_sample_manifest.json','*development*manifest*.csv','*development*manifest*.json'],
'schema':['channel_schema.json','tensor_schema.json','predictor_schema.json','*channel*schema*.json','*tensor*schema*.json','*predictor*schema*.json'],
'policy':['policy.json','decision_policy.json','registered_operating_point_comparison.json','*policy*.json','*operating*point*.json'],
'experiment_manifest':['experiment_manifest.json','*experiment*manifest*.json'],
'split_audit':['source_group_split_audit.csv','source_group_split_audit.json','*split*audit*.csv','*split*audit*.json'],
'oof_predictions':['topology_oof_predictions.csv','control_oof_predictions.csv','*oof*prediction*.csv','*oof*prediction*.parquet'],
'label_table':['*label*.csv','*label*.parquet'],
'checkpoint':['*.pt','*.pth','*.ckpt'],
'calibrator':['*calibrat*.joblib','*calibrat*.pkl','*calibrat*.pickle','*isotonic*.joblib','*isotonic*.pkl','*isotonic*.pickle','*calibrat*.json','*isotonic*.json'],
'ood':['*ood*.json','*ood*.joblib','*ood*.pkl','*distribution*.npz'],
'ensemble':['*ensemble*.json','*ensemble*.csv'],
'replay':['replay_test.json','*replay*.json','*replay*.npz']}
found={k:[] for k in patterns}
for root in V8_SEARCH_ROOTS:
    for role,pats in patterns.items():
        for pat in pats: found[role].extend(p for p in root.rglob(pat) if p.is_file())
for k in found: found[k]=dedupe(found[k])
rows=[]
for role,paths in found.items():
    for p in paths: rows.append({'role':role,'name':p.name,'path':str(p),'parent':str(p.parent),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
census=pd.DataFrame(rows); census.to_csv(DIRS['01_input_audit']/'v8_project_wide_artifact_census.csv',index=False)
report={'search_roots':[str(p) for p in V8_SEARCH_ROOTS],'counts':{k:len(v) for k,v in found.items()},'candidates':{k:[str(p) for p in v] for k,v in found.items()}}
atomic_json(DIRS['01_input_audit']/'artifact_discovery.json',report)
print(json.dumps(report['counts'],indent=2))


## 4. Table and artefact resolvers


In [ ]:
def load_json_table(path,keys=('pages','rows','assignments','fold_map')):
    obj=json.loads(Path(path).read_text())
    if isinstance(obj,list): return pd.DataFrame(obj)
    if isinstance(obj,dict):
        for k in keys:
            if isinstance(obj.get(k),list): return pd.DataFrame(obj[k])
        rows=[]
        for key,val in obj.items(): rows.append({'page':str(key),**val} if isinstance(val,dict) else {'page':str(key),'fold':val})
        return pd.DataFrame(rows)
    fail('json_table',f'Unsupported JSON structure: {path}')
def load_table(path):
    path=Path(path)
    if path.suffix.lower()=='.csv': return pd.read_csv(path)
    if path.suffix.lower()=='.parquet': return pd.read_parquet(path)
    if path.suffix.lower()=='.json': return load_json_table(path)
    fail('table_loader',f'Unsupported table: {path}')
ALIASES={'page_id':'page','page_name':'page','filename_stem':'page','group':'source_group','source':'source_group','score_group':'source_group','fold_id':'fold','outer_fold':'fold','test_fold':'fold'}
def choose_identical_file(paths,role,override=None,required=True):
    if override is not None:
        p=Path(override)
        if not p.exists(): fail('artifact_resolution',f'{role} override not found: {p}')
        return p
    paths=dedupe(paths)
    if not paths:
        if required: fail('artifact_resolution',f'No {role} found')
        return None
    if len(paths)==1: return paths[0]
    by_hash={}
    for p in paths: by_hash.setdefault(sha256_file(p),[]).append(p)
    if len(by_hash)==1:
        pref=[p for p in paths if 'v8_0d' in str(p).lower()] or paths
        print(role,'copies are byte-identical; using',pref[0]); return pref[0]
    fail('artifact_resolution',f'Multiple non-identical {role} candidates found',[str(p) for p in paths])


## 5. Load the PASSed fold audit and reconcile the development manifest


In [ ]:
if FOLD_AUDIT_RUN_OVERRIDE is not None:
    audit_runs=[Path(FOLD_AUDIT_RUN_OVERRIDE)]
else:
    if not FOLD_AUDIT_ROOT.exists(): fail('fold_map',f'Fold-map audit root not found: {FOLD_AUDIT_ROOT}')
    audit_runs=[p for p in FOLD_AUDIT_ROOT.iterdir() if p.is_dir() and (p/'fold_resolution.json').exists() and (p/'resolved_page_fold_map.csv').exists()]
if not audit_runs: fail('fold_map','No completed fold-map audit found',{'audit_root':str(FOLD_AUDIT_ROOT)})

valid=[]; invalid=[]
for audit_run in audit_runs:
    resolution_path=audit_run/'fold_resolution.json'; resolved_path=audit_run/'resolved_page_fold_map.csv'
    try:
        resolution=json.loads(resolution_path.read_text()); expected=resolution.get('resolved_page_csv_sha256'); actual=sha256_file(resolved_path)
        ok=resolution.get('status')=='PASS' and expected is not None and actual==expected
        rec={'audit_run':str(audit_run),'status':resolution.get('status'),'expected_sha256':expected,'actual_sha256':actual,'valid':bool(ok)}
        (valid if ok else invalid).append((audit_run,resolution,rec))
    except Exception as e: invalid.append((audit_run,None,{'audit_run':str(audit_run),'valid':False,'error':repr(e)}))
atomic_json(DIRS['01_input_audit']/'fold_audit_candidates.json',{'valid':[x[2] for x in valid],'invalid':[x[2] for x in invalid]})
if not valid: fail('fold_map','No PASSed fold-map audit has a matching resolved CSV hash',[x[2] for x in invalid])
FOLD_AUDIT_RUN,fold_resolution,_=max(valid,key=lambda x:x[0].stat().st_mtime)
FOLD_RESOLUTION_PATH=FOLD_AUDIT_RUN/'fold_resolution.json'; FOLD_MAP=FOLD_AUDIT_RUN/'resolved_page_fold_map.csv'
folds=pd.read_csv(FOLD_MAP)
required={'page','source_group','fold'}
if not required.issubset(folds.columns): fail('fold_map','Audited fold map missing required columns',{'required':sorted(required),'actual':list(folds.columns)})
folds=folds[['page','source_group','fold']].copy(); folds['page']=folds.page.astype(str).str.strip(); folds['source_group']=folds.source_group.astype(str).str.strip(); folds['fold']=folds.fold.astype(str).str.strip()
if folds.page.duplicated().any(): fail('fold_map','Audited page-level map contains duplicate pages',folds[folds.page.duplicated(False)].to_dict('records'))
for col,label in [('page','page'),('source_group','source group')]:
    q=folds.groupby(col).fold.nunique(); bad=q[q!=1]
    if len(bad): fail('fold_map',f'At least one {label} has multiple folds',bad.to_dict())
shutil.copy2(FOLD_RESOLUTION_PATH,DIRS['01_input_audit']/'fold_resolution.json'); folds.to_csv(DIRS['01_input_audit']/'resolved_page_fold_map.csv',index=False)

manifest_candidates=[p for p in found['development_manifest'] if 'development_score_manifest' in p.name.lower()] or found['development_manifest']
DEVELOPMENT_MANIFEST=choose_identical_file(manifest_candidates,'development manifest',DEVELOPMENT_MANIFEST_OVERRIDE,True)
manifest_raw=load_table(DEVELOPMENT_MANIFEST); manifest=manifest_raw.rename(columns={c:ALIASES[c] for c in manifest_raw.columns if c in ALIASES})
if not {'page','source_group'}.issubset(manifest.columns): fail('development_manifest','Manifest needs page and source_group',list(manifest.columns))
manifest['page']=manifest.page.astype(str).str.strip(); manifest['source_group']=manifest.source_group.astype(str).str.strip()
if manifest.page.duplicated().any() or manifest.source_group.isna().any(): fail('development_manifest','Duplicate page or missing source_group')
roster=folds.merge(manifest[['page','source_group']],on='page',how='outer',suffixes=('_fold_map','_manifest'),indicator=True); roster['source_group_match']=roster.source_group_fold_map==roster.source_group_manifest; roster.to_csv(DIRS['01_input_audit']/'development_manifest_fold_audit.csv',index=False)
if not roster._merge.eq('both').all(): fail('fold_map','Fold-map and development-manifest page sets differ',roster.loc[~roster._merge.eq('both')].to_dict('records'))
if not roster.source_group_match.all(): fail('fold_map','Fold-map and development-manifest source groups differ',roster.loc[~roster.source_group_match].to_dict('records'))
print('Fold audit run:',FOLD_AUDIT_RUN); print('Resolved fold map:',FOLD_MAP); print('Fold audit status:',fold_resolution['status']); print('Pages:',folds.page.nunique(),'Source groups:',folds.source_group.nunique(),'Folds:',folds.fold.nunique()); print('Roster/source-group audit: PASS')


## 6. Resolve schema, policy and fold artefacts


In [ ]:
SCHEMA_PATH=choose_identical_file(found['schema'],'schema',SCHEMA_OVERRIDE,True)
POLICY_PATH=choose_identical_file(found['policy'],'policy',POLICY_OVERRIDE,False)
REPLAY_PATH=choose_identical_file([p for p in found['replay'] if p.suffix.lower()=='.json'],'replay',REPLAY_OVERRIDE,False)
schema=json.loads(SCHEMA_PATH.read_text()); policy=json.loads(POLICY_PATH.read_text()) if POLICY_PATH else {'print_threshold':PRINT_THRESHOLD,'annotation_threshold':ANNOTATION_THRESHOLD,'print_annotation_margin':PRINT_ANNOTATION_MARGIN,'minimum_print_run':MIN_PRINT_RUN,'_source':'registered constants'}
def fold_token(path):
    for value in [path.stem,*[p.name for p in path.parents]]:
        m=re.search(r'(?:fold|outer|f)[_\-]?(\d+)',value.lower())
        if m: return str(int(m.group(1)))
    return None
def map_by_fold(paths,override,role):
    if override: return {str(k):Path(v) for k,v in override.items()}
    result={}; unresolved=[]
    for p in paths:
        tok=fold_token(p)
        if tok is None: unresolved.append(p); continue
        if tok in result and sha256_file(result[tok])!=sha256_file(p): fail('fold_artifacts',f'Multiple non-identical {role} artefacts for fold {tok}',[str(result[tok]),str(p)])
        result[tok]=p
    if unresolved:
        print('Unassigned',role,'files:')
        for q in unresolved: print(' ',q)
    return result
CHECKPOINT_BY_FOLD=map_by_fold(found['checkpoint'],CHECKPOINT_BY_FOLD_OVERRIDE,'checkpoint'); CALIBRATOR_BY_FOLD=map_by_fold(found['calibrator'],CALIBRATOR_BY_FOLD_OVERRIDE,'calibrator')
required_folds=set(folds.fold.astype(str)); missing_models=sorted(required_folds-set(CHECKPOINT_BY_FOLD)); missing_cal=sorted(required_folds-set(CALIBRATOR_BY_FOLD))
atomic_json(DIRS['01_input_audit']/'fold_artifact_mapping.json',{'models':{k:str(v) for k,v in CHECKPOINT_BY_FOLD.items()},'calibrators':{k:str(v) for k,v in CALIBRATOR_BY_FOLD.items()},'missing_models':missing_models,'missing_calibrators':missing_cal})
if missing_models or missing_cal: fail('fold_artifacts','Incomplete fold model/calibrator mapping',{'models':missing_models,'calibrators':missing_cal})


## 7. Validate registered constants and schema


In [ ]:
def get_any(d,names,default=None):
    for n in names:
        if n in d: return d[n]
    return default
ordered_channels=get_any(schema,['ordered_channels','channels','channel_names'])
if ordered_channels is None: fail('schema','No ordered channel list in schema')
if isinstance(ordered_channels,dict): ordered_channels=list(ordered_channels)
classes=[str(x).upper() for x in get_any(schema,['class_order','classes','ordered_classes'],['PRINT','ANNOTATION','UNCERTAIN'])]
if 'PRINT' not in classes or 'ANNOTATION' not in classes: fail('schema','Class order lacks PRINT/ANNOTATION',classes)
checks={'patch_size':int(get_any(schema,['patch_size','spatial_size','width'],PATCH_SIZE))==PATCH_SIZE,'print_threshold':float(get_any(policy,['print_threshold','threshold_print','p_print_threshold'],PRINT_THRESHOLD))==PRINT_THRESHOLD,'annotation_threshold':float(get_any(policy,['annotation_threshold','threshold_annotation','p_annotation_threshold'],ANNOTATION_THRESHOLD))==ANNOTATION_THRESHOLD,'margin':float(get_any(policy,['print_annotation_margin','class_margin','margin'],PRINT_ANNOTATION_MARGIN))==PRINT_ANNOTATION_MARGIN,'minimum_print_run':int(get_any(policy,['minimum_print_run','min_print_run','minimum_connected_run'],MIN_PRINT_RUN))==MIN_PRINT_RUN}
atomic_json(DIRS['01_input_audit']/'schema_policy_checks.json',checks)
if not all(checks.values()): fail('schema','Schema/policy values differ from registered constants',checks)
print('Channels:',ordered_channels); print('Classes:',classes)


## 8. Canonical frozen V7.8 reconstruction functions


In [ ]:
def immutable(original,candidate,mask):
    out=original.copy(); out[mask>0]=candidate[mask>0]
    if not np.array_equal(out[mask==0],original[mask==0]): fail('immutability','Outside-mask drift')
    return out
def background_surface(observed,mask,sigma=45.0):
    visible=(mask==0)&(observed>=200); values=observed.astype(np.float32)*visible; weights=visible.astype(np.float32); num=cv2.GaussianBlur(values,(0,0),sigma); den=cv2.GaussianBlur(weights,(0,0),sigma); surface=num/np.maximum(den,1e-4); fallback=cv2.GaussianBlur(observed,(0,0),sigma); return np.clip(np.where(den>0.02,surface,fallback),0,255).astype(np.uint8)
def support_maps(observed,mask):
    dark=(observed<190)&(mask==0); h=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0; v=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((15,1),np.uint8))>0; n,lab,stats,_=cv2.connectedComponentsWithStats(dark.astype(np.uint8),8); c=np.zeros_like(dark)
    for i in range(1,n):
        x,y,w,hg,a=stats[i]
        if 3<=a<=650 and w<=40 and hg<=40: c[lab==i]=True
    k=np.ones((2*BAND_WIDTH+1,2*BAND_WIDTH+1),np.uint8); return cv2.dilate(h.astype(np.uint8),k)>0,cv2.dilate(v.astype(np.uint8),k)>0,cv2.dilate(c.astype(np.uint8),k)>0
def geometry_t80(observed,mask,telea):
    paper=background_surface(observed,mask); normalised=cv2.divide(observed,np.maximum(paper,1),scale=255); horizontal,vertical,compact=support_maps(observed,mask); m=mask>0; preserve=m&(horizontal|vertical|compact)&(observed<200); remove=m&~preserve&(observed>=GEOMETRY_THRESHOLD)&(normalised>=GEOMETRY_THRESHOLD); ambiguous=m&~preserve&~remove; result=observed.copy(); result[remove]=paper[remove]; result[ambiguous]=telea[ambiguous]; result[preserve]=observed[preserve]; return immutable(observed,result,mask),{'preserve':preserve,'remove':remove,'ambiguous':ambiguous,'horizontal':horizontal,'vertical':vertical,'compact':compact,'paper':paper,'normalised':normalised}


## 9. Load cached development pages and require exact V7.8 equivalence


In [ ]:
status=pd.read_csv(V75/'tables'/'cache_status.csv'); cache_lookup={str(r.page):Path(r.cache) for _,r in status.iterrows()}; observed_map=image_map(ROOT/'aligned'); clean_map=image_map(ROOT/'source'/'clean'); fold_meta=folds.set_index('page'); manifest_meta=manifest.set_index('page')
pages=sorted(set(manifest.page)&set(cache_lookup)&set(observed_map)&set(clean_map))
if set(pages)!=set(manifest.page): fail('v78_equivalence','Not every development page resolves observed/cache/clean input',{'missing':sorted(set(manifest.page)-set(pages))})
runtime={}; eq=[]
for page in pages:
    obs=cv2.imread(str(observed_map[page]),0)
    with np.load(cache_lookup[page]) as z:
        required={'mask','telea','geometry_t80'}
        if not required.issubset(z.files): fail('v78_equivalence',f'{page}: cache arrays missing',sorted(required-set(z.files)))
        mask=z['mask']; telea=z['telea']; ref=z['geometry_t80']; built,routes=geometry_t80(obs,mask,telea); route_ok={}
        for k in ['preserve','remove','ambiguous','horizontal','vertical','compact']:
            rk='g80_'+k; route_ok[k]=(rk in z.files and np.array_equal(z[rk]>0,routes[k]))
    eq.append({'page':page,'geometry_exact':bool(np.array_equal(ref,built)),'different_pixels':int((ref!=built).sum()),'outside_immutable':bool(np.array_equal(ref[mask==0],obs[mask==0])),**{f'{k}_exact':v for k,v in route_ok.items()}})
    runtime[page]={'observed':obs,'mask':mask,'telea':telea,'v78':ref,'routes':routes,'clean_path':clean_map[page],'source_group':str(manifest_meta.loc[page,'source_group']),'fold_id':str(fold_meta.loc[page,'fold'])}
eq=pd.DataFrame(eq); eq.to_csv(DIRS['02_v78_equivalence']/'v78_equivalence.csv',index=False)
if not (eq.drop(columns=['page','different_pixels']).all().all() and eq.different_pixels.eq(0).all()): fail('v78_equivalence','Frozen V7.8 equivalence failed')
print('V7.8 exact on',len(eq),'pages')


## 10. Enumerate and lock every eligible pixel


In [ ]:
eligible_rows=[]
for page,z in runtime.items():
    candidate=(z['mask']>0)&(z['observed']<OBSERVED_DARK_THRESHOLD)&~z['routes']['preserve']; _,labels,_,_=cv2.connectedComponentsWithStats((z['mask']>0).astype(np.uint8),8); yy,xx=np.where(candidate)
    for y,x in zip(yy,xx): eligible_rows.append({'source_group':z['source_group'],'page':page,'fold_id':z['fold_id'],'y':int(y),'x':int(x),'observed_value':int(z['observed'][y,x]),'component_id':int(labels[y,x]),'v78_route':'remove' if z['routes']['remove'][y,x] else 'ambiguous'})
eligible=pd.DataFrame(eligible_rows).sort_values(['source_group','page','y','x']).reset_index(drop=True)
if eligible.empty: fail('eligibility','No eligible pixels')
eligible.to_parquet(DIRS['03_eligibility']/'eligible_pixel_manifest.parquet',index=False); eligible.groupby(['source_group','page','fold_id','v78_route']).size().rename('eligible_pixels').reset_index().to_csv(DIRS['03_eligibility']/'eligibility_summary.csv',index=False)
eligibility_lock={'rows':len(eligible),'semantic_sha256':object_hash(eligible[['source_group','page','fold_id','y','x']].to_dict('records')),'pages':eligible.page.nunique(),'source_groups':eligible.source_group.nunique()}; atomic_json(DIRS['03_eligibility']/'eligibility_hash.json',eligibility_lock); print(eligibility_lock)


## 11. Build the frozen 33×33 spatial channels


In [ ]:
def signed_mask_distance(mask):
    return cv2.distanceTransform((mask>0).astype(np.uint8),cv2.DIST_L2,5)-cv2.distanceTransform((mask==0).astype(np.uint8),cv2.DIST_L2,5)
def page_channels(z):
    obs=z['observed']; m=z['mask']; r=z['routes']; edges=cv2.Canny(obs,70,170).astype(np.float32)/255.0
    return {'observed_greyscale':obs.astype(np.float32)/255.0,'observed':obs.astype(np.float32)/255.0,'raw_greyscale':obs.astype(np.float32)/255.0,'background_normalised':r['normalised'].astype(np.float32)/255.0,'background_normalized':r['normalised'].astype(np.float32)/255.0,'normalised':r['normalised'].astype(np.float32)/255.0,'authorised_mask':(m>0).astype(np.float32),'authorized_mask':(m>0).astype(np.float32),'mask':(m>0).astype(np.float32),'mask_distance':np.clip(signed_mask_distance(m)/PATCH_RADIUS,-1,1).astype(np.float32),'horizontal_support':r['horizontal'].astype(np.float32),'vertical_support':r['vertical'].astype(np.float32),'compact_support':r['compact'].astype(np.float32),'preserve_route':r['preserve'].astype(np.float32),'remove_route':r['remove'].astype(np.float32),'ambiguous_route':r['ambiguous'].astype(np.float32),'observed_edges':edges,'edges':edges}
channel_cache={p:page_channels(z) for p,z in runtime.items()}; missing={p:sorted(set(ordered_channels)-set(ch)) for p,ch in channel_cache.items()}
if any(missing.values()): fail('tensor_schema','Runtime builder lacks frozen channels',missing)
padded_channels={p:{n:np.pad(ch[n],PATCH_RADIUS,mode='reflect') for n in ordered_channels} for p,ch in channel_cache.items()}
def tensor_for(page,y,x): return np.stack([padded_channels[page][n][y:y+PATCH_SIZE,x:x+PATCH_SIZE] for n in ordered_channels]).astype(np.float32)


## 12. Frozen model and calibrator adapters


In [ ]:
class CompactCNN(nn.Module):
    def __init__(self,in_channels,num_classes=3,widths=(16,32,64),dropout=0.15):
        super().__init__(); layers=[]; c=in_channels
        for w in widths: layers += [nn.Conv2d(c,w,3,padding=1),nn.BatchNorm2d(w),nn.ReLU(),nn.MaxPool2d(2)]; c=w
        self.features=nn.Sequential(*layers); self.head=nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Dropout(dropout),nn.Linear(c,num_classes))
    def forward(self,x): return self.head(self.features(x))
def load_model(path):
    try:
        m=torch.jit.load(str(path),map_location=DEVICE); m.eval(); return m
    except Exception: pass
    obj=torch.load(path,map_location=DEVICE,weights_only=False)
    if isinstance(obj,nn.Module): obj.to(DEVICE).eval(); return obj
    if isinstance(obj,dict):
        state=obj.get('state_dict',obj.get('model_state_dict',obj)); cfg=obj.get('model_config',obj.get('config',{})); m=CompactCNN(len(ordered_channels),int(cfg.get('num_classes',len(classes))),tuple(cfg.get('widths',[16,32,64])),float(cfg.get('dropout',0.15))); m.load_state_dict({k.replace('module.','').replace('model.',''):v for k,v in state.items()},strict=True); m.to(DEVICE).eval(); return m
    fail('model_loading',f'Unsupported checkpoint: {path}')
def load_calibrator(path):
    if path.suffix.lower() in ['.joblib','.pkl','.pickle']: return joblib.load(path)
    if path.suffix.lower()=='.json': return json.loads(path.read_text())
    fail('calibrator_loading',f'Unsupported calibrator: {path}')
def calibrate(cal,raw):
    raw=np.asarray(raw)
    if hasattr(cal,'predict_proba'): return np.asarray(cal.predict_proba(raw))
    if hasattr(cal,'predict'): return np.asarray(cal.predict(raw))
    if isinstance(cal,dict) and 'per_class' in cal:
        out=[np.interp(raw[:,i],cal['per_class'][c]['x'],cal['per_class'][c]['y']) for i,c in enumerate(classes)]; p=np.stack(out,1); return p/np.maximum(p.sum(1,keepdims=True),1e-12)
    if isinstance(cal,dict) and 'x' in cal and 'y' in cal and isinstance(cal['x'],dict):
        out=[np.interp(raw[:,i],cal['x'][c],cal['y'][c]) for i,c in enumerate(classes)]; p=np.stack(out,1); return p/np.maximum(p.sum(1,keepdims=True),1e-12)
    fail('calibrator_loading','Calibrator format cannot produce multiclass probabilities')
models={f:load_model(p) for f,p in CHECKPOINT_BY_FOLD.items()}; calibrators={f:load_calibrator(p) for f,p in CALIBRATOR_BY_FOLD.items()}; print('Loaded',len(models),'models and',len(calibrators),'calibrators')


## 13. Full-pixel held-out inference


In [ ]:
parts=[]; timing=[]
for fold,g0 in eligible.groupby('fold_id',sort=True):
    g=g0.reset_index(drop=True); model=models[str(fold)]; cal=calibrators[str(fold)]; start=time.perf_counter(); chunks=[]
    with torch.no_grad():
        for start_i in range(0,len(g),BATCH_SIZE):
            q=g.iloc[start_i:start_i+BATCH_SIZE]; arr=np.stack([tensor_for(r.page,int(r.y),int(r.x)) for _,r in q.iterrows()]); logits=model(torch.from_numpy(arr).to(DEVICE)); chunks.append(torch.softmax(logits,dim=1).cpu().numpy())
    raw=np.concatenate(chunks); cp=calibrate(cal,raw)
    if cp.ndim==1 or cp.shape!=raw.shape or not np.isfinite(cp).all(): fail('inference',f'Fold {fold}: calibrated probability shape/value failure',{'raw':raw.shape,'calibrated':cp.shape})
    q=g.copy()
    for i,c in enumerate(classes): q[f'raw_p_{c.lower()}']=raw[:,i]; q[f'p_{c.lower()}']=cp[:,i]
    parts.append(q); timing.append({'fold_id':fold,'eligible_pixels':len(q),'seconds':time.perf_counter()-start,'device':DEVICE})
pred=pd.concat(parts,ignore_index=True).sort_values(['source_group','page','y','x']).reset_index(drop=True); pd.DataFrame(timing).to_csv(DIRS['04_oof_predictions']/'fold_timing.csv',index=False); print('Scored',len(pred),'pixels')


## 14. Registered decision and connected-run rule


In [ ]:
pred['provisional']='UNCERTAIN'; pred['uncertain_reason']='below_threshold_or_margin'; print_ok=(pred.p_print>=PRINT_THRESHOLD)&((pred.p_print-pred.p_annotation)>=PRINT_ANNOTATION_MARGIN); annotation_ok=(pred.p_annotation>=ANNOTATION_THRESHOLD)&~print_ok; pred.loc[print_ok,['provisional','uncertain_reason']]=['PRINT','']; pred.loc[annotation_ok,['provisional','uncertain_reason']]=['ANNOTATION','']
if bool(get_any(policy,['ood_required'],False)) and not found['ood']: fail('decision_rule','Policy requires OOD but no OOD artefact was found')
if bool(get_any(policy,['ensemble_required','predictive_disagreement_required'],False)) and not found['ensemble']: fail('decision_rule','Policy requires ensemble disagreement but no ensemble definition was found')
pred['decision']=pred.provisional
for page,g in pred.groupby('page'):
    pm=np.zeros(runtime[page]['observed'].shape,bool); pm[g.y.astype(int),g.x.astype(int)]=g.provisional.eq('PRINT').to_numpy(); n,lab,stats,_=cv2.connectedComponentsWithStats(pm.astype(np.uint8),8); keep=np.zeros_like(pm)
    for cid in range(1,n):
        if stats[cid,cv2.CC_STAT_AREA]>=MIN_PRINT_RUN: keep[lab==cid]=True
    bad_coords=set(zip(*np.where(pm&~keep))); idx=g.index[g.apply(lambda r:(int(r.y),int(r.x)) in bad_coords,axis=1)]; pred.loc[idx,'decision']='UNCERTAIN'; pred.loc[idx,'uncertain_reason']='singleton_or_short_print_run'
pred.to_parquet(DIRS['04_oof_predictions']/'all_eligible_oof_predictions_pretruth.parquet',index=False); print(pred.decision.value_counts())


## 15. Assemble whole corrected pages and prediction lock


In [ ]:
page_hashes=[]
for page,g in pred.groupby('page'):
    z=runtime[page]; out=z['v78'].copy(); sel=g.decision.eq('PRINT'); yy=g.loc[sel,'y'].astype(int).to_numpy(); xx=g.loc[sel,'x'].astype(int).to_numpy(); out[yy,xx]=z['observed'][yy,xx]; pm=np.zeros(out.shape,bool); pm[yy,xx]=True
    if not np.array_equal(out[~pm],z['v78'][~pm]): fail('assembly',f'{page}: V7.8 changed outside PRINT')
    if not np.array_equal(out[z['mask']==0],z['observed'][z['mask']==0]): fail('assembly',f'{page}: outside-mask drift')
    d=DIRS['07_whole_pages']/page; d.mkdir(parents=True,exist_ok=False); cv2.imwrite(str(d/'observed.png'),z['observed']); cv2.imwrite(str(d/'v7_8.png'),z['v78']); cv2.imwrite(str(d/'v8_1.png'),out); cv2.imwrite(str(d/'authoritative_mask.png'),z['mask']); cv2.imwrite(str(d/'difference.png'),np.abs(out.astype(np.int16)-z['v78'].astype(np.int16)).astype(np.uint8)); decision=np.zeros(out.shape,np.uint8); decision[g.y.astype(int),g.x.astype(int)]=np.select([g.decision.eq('PRINT'),g.decision.eq('ANNOTATION')],[255,128],default=64).astype(np.uint8); cv2.imwrite(str(d/'decision_map.png'),decision); z['v81']=out
    page_hashes.append({'page':page,'observed_sha256':array_hash(z['observed']),'v78_sha256':array_hash(z['v78']),'v81_sha256':array_hash(out),'decision_sha256':array_hash(decision)})
pred_lock_path=DIRS['05_prediction_lock']/'all_eligible_oof_predictions.parquet'; pred.to_parquet(pred_lock_path,index=False); pd.DataFrame(page_hashes).to_csv(DIRS['05_prediction_lock']/'page_output_hashes.csv',index=False)
lock={'created_utc':datetime.now(timezone.utc).isoformat(),'fold_map':str(FOLD_MAP),'fold_resolution_sha256':sha256_file(FOLD_RESOLUTION_PATH),'fold_map_semantic_sha256':object_hash(folds.sort_values('page').to_dict('records')),'development_manifest':str(DEVELOPMENT_MANIFEST),'schema_sha256':sha256_file(SCHEMA_PATH),'policy_sha256':sha256_file(POLICY_PATH) if POLICY_PATH else None,'eligibility_sha256':eligibility_lock['semantic_sha256'],'predictions_sha256':sha256_file(pred_lock_path),'page_hashes_sha256':sha256_file(DIRS['05_prediction_lock']/'page_output_hashes.csv'),'checkpoints':{f:sha256_file(p) for f,p in CHECKPOINT_BY_FOLD.items()},'calibrators':{f:sha256_file(p) for f,p in CALIBRATOR_BY_FOLD.items()},'clean_loaded':False}; atomic_json(DIRS['05_prediction_lock']/'prediction_lock.json',lock); print('Prediction lock written')


## 16. Clean-truth scoring after lock


In [ ]:
def printed_ink(im): return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
auth=None
for lp in found['label_table']:
    try:
        q=load_table(lp).rename(columns={c:ALIASES[c] for c in load_table(lp).columns if c in ALIASES}); truth_col=next((c for c in ['truth','label','target','class'] if c in q.columns),None)
        if truth_col and {'page','y','x'}.issubset(q.columns):
            qq=q[['page','y','x',truth_col]].copy().rename(columns={truth_col:'truth'}); qq['page']=qq.page.astype(str)
            if not qq.duplicated(['page','y','x']).any() and len(qq.merge(pred[['page','y','x']],on=['page','y','x'],how='inner'))>0: auth=qq; print('Using durable label table:',lp); break
    except Exception: pass
if auth is not None:
    pred=pred.merge(auth,on=['page','y','x'],how='left',validate='one_to_one'); pred['truth']=pred.truth.replace({0:'ANNOTATION',1:'PRINT',2:'UNCERTAIN'}).fillna('UNCERTAIN').astype(str).str.upper()
else:
    truth=[]
    for r in pred.itertuples():
        z=runtime[r.page]
        if 'clean_ink' not in z: z['clean_ink']=printed_ink(cv2.imread(str(z['clean_path']),0))
        ink=z['clean_ink']; y,x=int(r.y),int(r.x); local=ink[max(0,y-1):min(ink.shape[0],y+2),max(0,x-1):min(ink.shape[1],x+2)]; truth.append('PRINT' if local.any() else 'ANNOTATION')
    pred['truth']=truth
pred['extra_true']=(pred.decision=='PRINT')&(pred.truth=='PRINT'); pred['extra_false']=(pred.decision=='PRINT')&(pred.truth=='ANNOTATION'); pred['truth_uncertain_print']=(pred.decision=='PRINT')&(pred.truth=='UNCERTAIN'); pred.to_parquet(DIRS['06_truth_scoring']/'all_eligible_oof_predictions_scored.parquet',index=False)


## 17. Metrics, catastrophic scan and clustered uncertainty


In [ ]:
def summarize(g):
    t=int(g.extra_true.sum()); f=int(g.extra_false.sum()); u=int(g.truth_uncertain_print.sum()); added=int((g.decision=='PRINT').sum()); return pd.Series({'eligible':len(g),'print':added,'annotation':int((g.decision=='ANNOTATION').sum()),'uncertain':int((g.decision=='UNCERTAIN').sum()),'extra_true':t,'extra_false':f,'truth_uncertain_print':u,'false_true_ratio':f/max(t,1),'false_preservation_rate':f/max(added,1),'utility_before_catastrophe':t-5*f,'print_coverage':added/max(len(g),1),'uncertain_rate':float((g.decision=='UNCERTAIN').mean())})
page=pred.groupby(['source_group','page']).apply(summarize,include_groups=False).reset_index(); group=pred.groupby('source_group').apply(summarize,include_groups=False).reset_index(); pooled=summarize(pred)
cat=[]
for page_id,g in pred.groupby('page'):
    for cid,c in g.groupby('component_id'):
        t=int(c.extra_true.sum()); f=int(c.extra_false.sum())
        if f>=50 or (f/max(t,1)>1 and f>=20): cat.append({'page':page_id,'source_group':c.source_group.iloc[0],'component_id':int(cid),'extra_true':t,'extra_false':f,'false_true_ratio':f/max(t,1)})
cat=pd.DataFrame(cat,columns=['page','source_group','component_id','extra_true','extra_false','false_true_ratio']); cat.to_csv(DIRS['06_truth_scoring']/'catastrophic_components.csv',index=False); cat_by_page=cat.groupby('page').size().to_dict() if len(cat) else {}; page['catastrophic_components']=page.page.map(cat_by_page).fillna(0).astype(int); page['utility']=page.extra_true-5*page.extra_false-500*page.catastrophic_components; pooled['catastrophic_components']=len(cat); pooled['utility']=int(pooled.extra_true)-5*int(pooled.extra_false)-500*len(cat); page.to_csv(DIRS['06_truth_scoring']/'results_by_page.csv',index=False); group.to_csv(DIRS['06_truth_scoring']/'results_by_source_group.csv',index=False)
rng=np.random.default_rng(BOOTSTRAP_SEED); unique_pages=page.page.to_numpy(); boot=[]
for _ in range(BOOTSTRAP_SAMPLES):
    sample=rng.choice(unique_pages,len(unique_pages),replace=True); q=pd.concat([page[page.page.eq(p)] for p in sample],ignore_index=True); t=int(q.extra_true.sum()); f=int(q.extra_false.sum()); added=int(q['print'].sum()); cats=int(q.catastrophic_components.sum()); boot.append((f/max(t,1),f/max(added,1),t-5*f-500*cats))
boot=np.asarray(boot); unc={'false_true_ci95':np.percentile(boot[:,0],[2.5,97.5]).tolist(),'false_preservation_ci95':np.percentile(boot[:,1],[2.5,97.5]).tolist(),'utility_ci95':np.percentile(boot[:,2],[2.5,97.5]).tolist()}; atomic_json(DIRS['06_truth_scoring']/'clustered_uncertainty.json',unc); atomic_json(DIRS['06_truth_scoring']/'pooled_metrics.json',pooled.to_dict()); print(pooled.to_dict()); print(unc)


## 18. Registered gate status and visual queue


In [ ]:
benefit_pages=int((page.extra_true>=10).sum()); benefit_groups=int(page.loc[page.extra_true>0,'source_group'].nunique()); gates={'D1_integrity':True,'D2_immutability':True,'D3_catastrophic_safety':len(cat)==0,'D4_false_preservation':float(pooled.false_true_ratio)<=0.10 and unc['false_true_ci95'][1]<=0.20,'D5_pixel_safety':float(pooled.false_preservation_rate)<=0.005 and unc['false_preservation_ci95'][1]<=0.01,'D6_positive_utility':float(pooled.utility)>0 and unc['utility_ci95'][0]>0,'D7_non_trivial_benefit':int(pooled.extra_true)>=500 and benefit_pages>=10 and benefit_groups>=3,'D8_source_stability':bool((group.false_true_ratio<=0.25).all() and len(cat)==0),'D9_sparse_hard_negative':None,'D10_uncertainty_behaviour':float(pooled.uncertain_rate)<=0.95,'D11_baseline_dominance':None,'D12_visual_audit':False}; first_failed=next((k for k,v in gates.items() if v is False and k!='D12_visual_audit'),None); verdict={'gates':gates,'first_failed_gate':first_failed,'freeze_permitted':False,'note':'Human visual audit remains incomplete; D9/D11 require explicit historic evidence.'}; atomic_json(OUT/'v8_1_verdict.json',verdict)
false_q=page.sort_values(['extra_false','false_true_ratio'],ascending=False).head(50).assign(queue='false_heavy'); benefit_q=page.sort_values('extra_true',ascending=False).head(50).assign(queue='beneficial'); unc_q=page.sort_values('uncertain',ascending=False).head(50).assign(queue='uncertain'); queue=pd.concat([false_q,benefit_q,unc_q],ignore_index=True).drop_duplicates(['queue','page']); queue['review_status']='unreviewed'; queue['notation_preserved']=''; queue['annotation_retained']=''; queue['difference_irrelevant']=''; queue['reviewer_comment']=''; queue.to_csv(DIRS['08_visual_audit']/'visual_audit_queue.csv',index=False)
for page_id in runtime:
    z=runtime[page_id]; fig,ax=plt.subplots(1,3,figsize=(18,10))
    for a,im,title in zip(ax,[z['v78'],z['v81'],np.abs(z['v81'].astype(np.int16)-z['v78'].astype(np.int16))],['Frozen V7.8','V8.1 full-pixel','Absolute difference']): a.imshow(im,cmap='gray',vmin=0,vmax=255); a.set_title(title); a.axis('off')
    plt.tight_layout(); plt.savefig(DIRS['08_visual_audit']/f'{page_id}_whole_page_comparison.png',dpi=140,bbox_inches='tight'); plt.close(fig)
print(json.dumps(verdict,indent=2))


## 19. Deterministic replay and final manifest


In [ ]:
sample=pred.head(min(512,len(pred))).copy(); outputs=[]
for repeat in range(2):
    allp=[]
    for fold,g in sample.groupby('fold_id',sort=True):
        arr=np.stack([tensor_for(r.page,int(r.y),int(r.x)) for _,r in g.iterrows()])
        with torch.no_grad(): raw=torch.softmax(models[str(fold)](torch.from_numpy(arr).to(DEVICE)),dim=1).cpu().numpy()
        allp.append(calibrate(calibrators[str(fold)],raw))
    outputs.append(np.concatenate(allp))
replay={'rows':len(sample),'exact_calibrated_probability_replay':bool(np.array_equal(outputs[0],outputs[1])),'first_sha256':array_hash(outputs[0]),'second_sha256':array_hash(outputs[1])}; atomic_json(DIRS['09_replay']/'same_run_replay.json',replay)
if not replay['exact_calibrated_probability_replay']: fail('replay','Same-run calibrated replay differed',replay)
software={'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform(),'device':DEVICE}; manifest_out={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'8.1-full-pixel-whole-page-oof-audited-fold-map','v75_run':str(V75),'fold_audit_run':str(FOLD_AUDIT_RUN),'fold_resolution':str(FOLD_RESOLUTION_PATH),'fold_map':str(FOLD_MAP),'development_manifest':str(DEVELOPMENT_MANIFEST),'schema':str(SCHEMA_PATH),'policy':str(POLICY_PATH) if POLICY_PATH else None,'registered_constants':{'patch_size':PATCH_SIZE,'observed_dark_threshold':OBSERVED_DARK_THRESHOLD,'print_threshold':PRINT_THRESHOLD,'annotation_threshold':ANNOTATION_THRESHOLD,'margin':PRINT_ANNOTATION_MARGIN,'minimum_print_run':MIN_PRINT_RUN},'eligibility':eligibility_lock,'pooled_metrics':pooled.to_dict(),'gates':gates,'clean_loaded_only_after_prediction_lock':True,'independent_data_used':False,'model_trained':False,'calibration_fitted':False,'threshold_changed':False,'software':software}; atomic_json(OUT/'experiment_manifest.json',manifest_out)
summary=['V8.1 FULL-PIXEL WHOLE-PAGE OOF',f'Output: {OUT}',f'Fold audit: {FOLD_AUDIT_RUN}',f'Pages: {eligible.page.nunique()}',f'Eligible pixels: {len(eligible)}',f'Final PRINT: {int((pred.decision=="PRINT").sum())}',f'Extra true print: {int(pred.extra_true.sum())}',f'Extra false annotation: {int(pred.extra_false.sum())}',f'Truth-UNCERTAIN PRINT: {int(pred.truth_uncertain_print.sum())}',f'Catastrophic components: {len(cat)}',f'First failed gate: {first_failed}',f'Visual audit: pending']; (OUT/'executive_summary.txt').write_text(chr(10).join(summary)); print(chr(10).join(summary)); print('COMPLETE:',OUT)


## Outputs to bring back after the run

- `executive_summary.txt`
- `v8_1_verdict.json`
- `experiment_manifest.json`
- `01_input_audit/fold_audit_candidates.json`
- `01_input_audit/resolved_page_fold_map.csv`
- `01_input_audit/fold_resolution.json`
- `01_input_audit/development_manifest_fold_audit.csv`
- `01_input_audit/fold_artifact_mapping.json`
- `06_truth_scoring/results_by_page.csv`
- `06_truth_scoring/results_by_source_group.csv`
- `06_truth_scoring/catastrophic_components.csv`
- `06_truth_scoring/clustered_uncertainty.json`
- `08_visual_audit/visual_audit_queue.csv`
- `09_replay/same_run_replay.json`

If the notebook stops, bring back the JSON under `failures/` plus `01_input_audit/v8_project_wide_artifact_census.csv`.
